In [ ]:
from lire_fichiers import charger_donnees

donnees = charger_donnees()

# Affiche la liste des fichiers chargés et leurs dimensions
for nom, table in donnees.items():
    print(f"{nom} : {table.shape[0]} lignes, {table.shape[1]} colonnes")

# Manipule un fichier précis par son nom
premier_fichier = list(donnees.keys())[1]
df_1 = donnees[premier_fichier]
df_1.head()

In [ ]:
import pandas as pd

# 1. Calcul des valeurs manquantes sur l'ensemble des fichiers de 'donnees'
total_lignes = sum(len(table) for table in donnees.values())

valeurs_valides = pd.concat(
    [table.notna().sum() for table in donnees.values()], axis=1
).sum(axis=1)

bilan_nan = pd.DataFrame({
    "nb_nan": (total_lignes - valeurs_valides).astype(int),
    "pourcentage_nan": ((total_lignes - valeurs_valides) / total_lignes * 100).round(2),
}).sort_values(by="nb_nan", ascending=False)

# 2. Extraction des colonnes 100 % vides
colonnes_vides = list(bilan_nan[bilan_nan["pourcentage_nan"] == 100].index)
print(f"Colonnes complètement vides ({len(colonnes_vides)}) :", colonnes_vides)

# Affiche le tableau complet trié des colonnes les plus vides aux plus remplies
bilan_nan

NameError: name 'donnees' is not defined

In [1]:
import fsspec
import pyarrow.parquet as pq

URL_ENTREPRISES = (
    "https://minio.lab.sspcloud.fr/k1000ensae/diffusion/projet-2a/data/"
    "donnees_financieres_entreprises.parquet"
)

with fsspec.open(URL_ENTREPRISES, "rb") as f:
    pf = pq.ParquetFile(f)
    print("Colonnes exactes :", pf.schema_arrow.names)
    apercu = next(pf.iter_batches(batch_size=5)).to_pandas()

apercu

Colonnes exactes : ['siren', 'date_cloture_exercice', 'type_bilan', 'confidentiality', 'liasse']


,siren,date_cloture_exercice,type_bilan,confidentiality,liasse
0,005420120,2018-12-31,C,Public,"[(DJ, 1064), (DP, 48780), (FW, 385776), (LN, 5..."
1,005420120,2021-12-31,C,Public,"[(LS, 348019), (1A, 3546013), (AW, 0), (BB-BC,..."
2,005420120,2017-12-31,C,Public,"[(GM, 296672), (HH, 260060), (DA, 711840), (FH..."
3,005420120,2016-12-31,C,Public,"[(0G, 69830594), (BF-BG, 1760), (FS, 11836), (..."
4,005420120,2019-12-31,C,Public,"[(AS, 84588), (FD, 0), (FH, 0), (GP, 805059), ..."


In [2]:
import fsspec
import pyarrow.parquet as pq

URL_ENTREPRISES = (
    "https://minio.lab.sspcloud.fr/k1000ensae/diffusion/projet-2a/data/"
    "donnees_financieres_entreprises.parquet"
)

with fsspec.open(URL_ENTREPRISES, "rb") as f:
    pf = pq.ParquetFile(f)
    echantillon = next(pf.iter_batches(batch_size=1000)).to_pandas()

# Convertit la colonne liasse en vrais dictionnaires Python
echantillon["liasse"] = echantillon["liasse"].apply(dict)

# Garde uniquement les lignes dont la liasse n'est pas vide (bilans non confidentiels)
non_vides = echantillon[echantillon["liasse"].str.len() > 0]

print("Répartition des types de bilan :")
print(non_vides["type_bilan"].value_counts(), "\n")

# Affiche le contenu de la liasse pour le premier bilan complet ('C') et simplifié ('S')
for tb in ["C", "S"]:
    sous_ensemble = non_vides[non_vides["type_bilan"] == tb]
    if not sous_ensemble.empty:
        print(f"Exemple liasse type '{tb}' :", sous_ensemble["liasse"].iloc[0])

Répartition des types de bilan :
type_bilan
C    963
K     21
S     16
Name: count, dtype: int64 

Exemple liasse type 'C' : {'DJ': 1064, 'DP': 48780, 'FW': 385776, 'LN': 5850813, 'AJ': 540293, 'BI': 0, 'FD': 954, 'LS': 1503473, 'NK': 78273512, 'AP': 3174799, 'DQ': 2839131, 'DX': 148218, 'FE': 0, 'FY': 559989, 'HB': 209176, 'HI': 481014, 'QU': 2368581, 'AU': 196924, 'DB': 7963, 'CI': 0, 'CK': 2005218, 'CG': 0, 'CH': 36923, 'BB-BC': 9347425, 'CD-CE': 12917722, 'DD': 71183, 'EA': 123502, 'GG': -1281259, 'GJ': 39482, 'AN': 2288972, 'AT-AU': 47706, 'UC': 12390, 'GM': 58113, 'ØR': 2563605, 'CJ': 17122824, 'DG': 89549282, 'EC': 4341098, 'FL': 135797, 'AS': 80009, 'CA': 1703757, 'HG': 9499, 'LO': 0, 'BH-BI': 2559, 'CE': 301461, 'DI': -289131, 'FQ': 206166, 'HA': 280517, 'KE': 0, 'QW': 57930, 'AN-AO': 1888651, 'CF': 1404319, 'GK': 28079, '7C': 220031, 'FK': 0, 'HF': 111571, 'ØK': 82226, 'BZ-CA': 529304, 'GP': 745630, 'CO-1A': 97498351, 'CR': 1299000, 'PF': 27044, 'BV': 199270, 'CH-CI': 36923, 